# H01 — 把虚拟按钮换成真实 GPIO

**学习入口：把下面 prompt 发给 VS Code Codex 侧边栏。** Notebook 也可自行逐格阅读。
C++ 在持续 ESP-IDF 工程编辑与构建；Notebook 运行仅显示示例/手动证据卡。
**状态：Alpha；固件构建与实机验收分开记录，见 ../VALIDATION.md。**

```text
开始 H01 — 把虚拟按钮换成真实 GPIO。
请先读取本工程 AGENTS.md、.academy/course.json 和 .academy/progress.md。
通过 course_root 找到 step2/course.json 和 step2/notebooks/H01-gpio.ipynb。
读取 Notebook 的 cells/source 与 academy.step_id/mode；先读当前步骤相关格。
检查工程、环境和前置条件，从第一个未完成步骤开始；每轮只推进一个小步骤。
Guided 可合并指定示例并说明 diff；Modify/Debug/Build/Open 先让我尝试，按需逐级提示。
不要主动读取 solution.md 或其他答案。环境故障可以排查修复，解释后回到原步骤。
不要把挑战 Bug 自动修掉。以当前代码与真实证据验收，记录提示级别和下一步。
我已有一定 C++ 基础，Python 几乎零基础。现在开始。
```

不用 agent 时按步骤标题顺序操作；卡住后再逐级查看本课 hints.md。启动 prompt 的路径按工作区课程链接解析，不靠其他窗口或聊天记忆。

## 我们要解决什么问题？

STEP 1 的“按下”是一个软件输入。现在我们要验证真实按钮的电平、真实 LED 的电流回路，而不把串口状态当作灯真的亮了。

前置：H00 的构建、烧录和启动日志通过；LED、电阻、按钮、面包板。

## 本次够用的知识

GPIO 编号不是排针第几孔。输入上拉使松开时接近 HIGH，按钮把输入接到 GND 后读 LOW；输出 HIGH 经限流电阻驱动 LED。先按住亮、松开灭，再迁移自己的按键事件 Controller，抖动放到 H03 处理。

先预测、运行和记录，再补概念。每节开始保存现有工程差异，后续沿用同一工程。

## 可选：Notebook 示例与证据卡

使用课程 .venv kernel 运行下格；这只是显示工具，不需要学习 Python。侧边栏学习不依赖这些格，build/flash/monitor 都在 ESP-IDF Terminal 执行。

In [ ]:
from pathlib import Path
import sys
import json

course_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "step2/course.json").is_file():
        course_root = candidate
        break
    link = candidate / ".academy/course.json"
    if link.is_file():
        course_root = Path(json.loads(link.read_text(encoding="utf-8"))["course_root"])
        break
if course_root is None or not (course_root / "step2/course.json").is_file():
    raise RuntimeError("请从课程工作区运行，或先在学习工程 .academy/course.json 链接课程根目录")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from tools.esp32_notebook import install, hardware_checklist
install(get_ipython())


## 断电接线并解释回路 · Guided

使用经典 ESP32 基线：GPIO18 → 330 Ω 电阻 → LED 阳极，LED 阴极 → GND；GPIO19 → 按钮 → GND，输入使用内部上拉。
断电检查四脚按钮同侧是否内部相连、LED 极性、共地和电阻。不要把按钮接到 5 V。按约 2 V 的 LED 压降估算 (3.3-2)/330 A，说明它只是估算。完整图见 ../BOARD.md；核对实物 pinout。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 读原始电平，再驱动灯 · Guided

把示例增量合并到引导工程 main/main.cpp。main/CMakeLists.txt 的 REQUIRES 增加 esp_driver_gpio；只保留一个 app_main()。
运行 build/flash/monitor。先不按按钮：预测 raw、pressed 与 LED；再按住、松开。日志只有变化时输出，不以打印数量作为按键次数。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

示例来源：`step2/lessons/H01/examples/gpio_led.cpp`。目标：`main/main.cpp`。这是教学示例；合并后以工程源码为准，后续不要来回修改两份实现。依赖参考：`REQUIRES esp_driver_gpio freertos log`；SRCS 按本步骤目标文件登记。

In [ ]:
%%esp32_example --target main/main.cpp
#include "driver/gpio.h"
#include "esp_log.h"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

// GPIO labels, not physical header positions. Classic ESP32 external LED/button.
constexpr gpio_num_t kLed = GPIO_NUM_18;
constexpr gpio_num_t kButton = GPIO_NUM_19;

extern "C" void app_main() {
    gpio_config_t output{};
    output.pin_bit_mask = 1ULL << kLed;
    output.mode = GPIO_MODE_OUTPUT;
    ESP_ERROR_CHECK(gpio_config(&output));
    ESP_ERROR_CHECK(gpio_set_level(kLed, 0));
    gpio_config_t input{};
    input.pin_bit_mask = 1ULL << kButton;
    input.mode = GPIO_MODE_INPUT;
    input.pull_up_en = GPIO_PULLUP_ENABLE;
    ESP_ERROR_CHECK(gpio_config(&input));
    int previous = -1;
    while (true) {
        const int raw = gpio_get_level(kButton);
        const bool pressed = raw == 0;
        ESP_ERROR_CHECK(gpio_set_level(kLed, pressed ? 1 : 0));
        if (raw != previous) {
            ESP_LOGI("gpio", "raw=%d pressed=%d output=%d", raw, pressed, pressed);
            previous = raw;
        }
        vTaskDelay(pdMS_TO_TICKS(10));
    }
}


## 更换观察方法 · Modify

先把 LED 逻辑反相，再恢复。记录日志 output 与真实灯是否一致。拔下 USB 后仅拔掉 LED 支路、重新上电：程序还能上报 output=1 吗？
这证明输出寄存器状态与实物发光是两种证据。每次改线都先断电。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 按钮反了还是浮空了 · Debug

只将 pressed 的比较从 raw == 0 改成 raw != 0，观察松开/按下日志并定位极性错误，之后恢复。
第二个问题只做推理：若关闭上拉且按钮松开，GPIO 是否必然读 0？不要把偶然稳定值当作浮空输入的保证。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 同一行为，两种入口 · Guided

ESP-IDF 中 app_main() 可以创建任务，也可以在可让出 CPU 的循环中处理输入；Arduino 中初始化常放 setup()，周期处理放 loop()。
对照 digitalRead()/digitalWrite() 与 gpio_get_level()/gpio_set_level()，画出 Controller → Board → Driver。已有 Arduino 库与快速原型可支持 Arduino；组件、配置、RTOS 和 OTA 需求支持选择 ESP-IDF。此处不另装一套环境。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

### 迁移桥梁：只替换 Board 实现

下面保留 STEP 1 的 Board 接口，换成真实 GPIO 实现。board.hpp 与原 include/academy/board.hpp
是同一契约；如果已复制原路径，调整 include 并只保留一份 Board 定义。
把头文件放 main，hal_entry.cpp 的入口合并到 main/main.cpp，不并列编译多个 app_main。
下一步由你接自己的 Controller；这里没有给出单次按键事件或消抖答案。


示例：`step2/lessons/H01/examples/board.hpp` → `main/board.hpp`。SRCS 只登记实际 .cpp；REQUIRES esp_driver_gpio freertos log。

In [ ]:
%%esp32_example --target main/board.hpp
#pragma once

namespace academy {

    enum class Level {
        low,
        high
    };

    // Same contract as STEP 1. LOW means pressed; configure before first read.
    class Board {
    public:
        virtual ~Board() = default;
        virtual Level read_button() const = 0;
        virtual void write_led(bool on) = 0;
    };

} // namespace academy


示例：`step2/lessons/H01/examples/esp32_board.hpp` → `main/esp32_board.hpp`。SRCS 只登记实际 .cpp；REQUIRES esp_driver_gpio freertos log。

In [ ]:
%%esp32_example --target main/esp32_board.hpp
#pragma once
#include "board.hpp"
#include "driver/gpio.h"

namespace academy {

    class Esp32Board final : public Board {
    public:
        Esp32Board(gpio_num_t button, gpio_num_t led) : button_(button), led_(led) {
        }

        esp_err_t init() {
            gpio_config_t output{};
            output.pin_bit_mask = 1ULL << led_;
            output.mode = GPIO_MODE_OUTPUT;
            esp_err_t result = gpio_config(&output);
            if (result != ESP_OK) {
                return result;
            }
            result = gpio_set_level(led_, 0);
            if (result != ESP_OK) {
                return result;
            }
            gpio_config_t input{};
            input.pin_bit_mask = 1ULL << button_;
            input.mode = GPIO_MODE_INPUT;
            input.pull_up_en = GPIO_PULLUP_ENABLE;
            return gpio_config(&input);
        }
        Level read_button() const override {
            return gpio_get_level(button_) == 0 ? Level::low : Level::high;
        }
        void write_led(bool on) override {
            // Same void contract as STEP 1; configuration errors are checked by init.
            ESP_ERROR_CHECK(gpio_set_level(led_, on ? 1 : 0));
        }

    private:
        gpio_num_t button_;
        gpio_num_t led_;
    };

} // namespace academy


示例：`step2/lessons/H01/examples/hal_entry.cpp` → `main/main.cpp`。SRCS 只登记实际 .cpp；REQUIRES esp_driver_gpio freertos log。

In [ ]:
%%esp32_example --target main/main.cpp
#include "esp32_board.hpp"
#include "freertos/FreeRTOS.h"
#include "freertos/task.h"

extern "C" void app_main() {
    academy::Esp32Board board(GPIO_NUM_19, GPIO_NUM_18);
    ESP_ERROR_CHECK(board.init());
    while (true) {
        // Guided level-following only. Your edge/debounce Controller comes next.
        board.write_led(board.read_button() == academy::Level::low);
        vTaskDelay(pdMS_TO_TICKS(10));
    }
}


## 迁移软件契约 · Build

复用 STEP 1 的 Board 接口和你自己的 Controller；实现 Esp32Board，配置/极性放在硬件层。行为要求：按下形成一个业务事件，持续按住不能每轮重复触发，松开可再次触发。
本课先接受抖动可能产生额外边沿，把观察记录留到 H03；不要暗用 STEP 1 Solution。用 fake Board 验证逻辑，用实物验证极性。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 设计一颗状态灯 · Open

定义启动、运行、故障三种灯光语义，说明用户怎样区分；暂不实现复杂闪烁。拍一张带 GPIO 标注的接线照片并填写实物清单。
H02 会创建自己的最小工程；把本课行为和接线记录作为迁移依据。

**停在这里：** 完成本步骤后记录预期、实际和证据，再进入下一步；agent 等待你的结果。

## 验收与复盘

- [ ] 断电核对 GPIO18/19、3.3 V、GND、LED 电阻和极性
- [ ] raw 松开为 1、按下为 0，日志与实物分别记录
- [ ] HAL 中保留极性，上层不直接调用 GPIO API

分别记录 host、firmware build、flash、日志、实物与网络往返；未知保持未确认。手动卡片只是学习者记录，不自动测量，也不自动写入进度文件。

复盘：我选择了什么机制？哪个证据排除了错误假设？哪些内容还需要测量？把当前 Step、Hint level、代码版本、证据和下次起点写入 `.academy/progress.md`。

In [ ]:
hardware_checklist(
    'H01',
    [
        '断电核对 GPIO18/19、3.3 V、GND、LED 电阻和极性',
        'raw 松开为 1、按下为 0，日志与实物分别记录',
        'HAL 中保留极性，上层不直接调用 GPIO API',
    ],
)


## 遇到困难时

先完成一次尝试，再打开 `step2/lessons/H01/hints.md`，每次只读当前步骤的一层。完整参考与解题分析单独保存，默认路径不主动读取。

## 官方版本参考

- [api-reference/peripherals/gpio.html](https://docs.espressif.com/projects/esp-idf/en/v5.5.1/esp32/api-reference/peripherals/gpio.html)